# Concrete strength prediction

This notebook predicts concrete strength for the Playground Series Season 3 Episode 9 competition:
- Feature engineering is derived from the [EDA which makes sense](https://www.kaggle.com/code/ambrosm/pss3e9-eda-which-makes-sense).
- `AgeInDays` is target-encoded as if it were a categorical variable.
- Some models perform slightly better if we eliminate `FlyAshComponent`, `WaterComponent` and`FineAggregateComponent`.
- The original data is not used.
- The final model is a blend of `GradientBoostingRegressor`, `LGBMRegressor`, `CatBoostRegressor`, `RandomForestRegressor` and `Ridge`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm, xgboost, catboost

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

np.set_printoptions(linewidth=150, edgeitems=5)


In [ ]:
result_list = []
train = pd.read_csv('/kaggle/input/playground-series-s3e9/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/playground-series-s3e9/test.csv', index_col='id')
original = pd.read_csv('/kaggle/input/predict-concrete-strength/ConcreteStrengthData.csv')
original.rename(columns={"CementComponent ": "CementComponent"}, inplace=True)
train['original'] = False
test['original'] = False
original['original'] = True

target = 'Strength'
original_features = list(test.columns)

print(f"Length of train: {len(train)}")
print(f"Length of test:  {len(test)}")
print(f"Length of original: {len(original)}")
print()

temp1 = train.isna().sum().sum()
temp2 = test.isna().sum().sum()
if temp1 == 0 and temp2 == 0:
    print('There are no null values in train and test.')
else:
    print(f'There are {temp1} null values in train')
    print(f'There are {temp2} null values in train')
print()

print('Sample lines from train:')
train.tail(3)

# Feature engineering

In [ ]:
for df in [train, test, original]:
#     df['Water_Cement'] = df['WaterComponent']/df['CementComponent'] # useless
#     df['Aggregate'] = df['CoarseAggregateComponent'] + df['FineAggregateComponent'] # useless
#     df['Aggregate_Cement'] = df['Aggregate']/df['CementComponent'] # useless
#     df['Slag_Cement'] = df['BlastFurnaceSlag']/df['CementComponent'] # useless
#     df['Ash_Cement'] = df['FlyAshComponent']/df['CementComponent'] # useless
#     df['Plastic_Cement'] = df['SuperplasticizerComponent']/df['CementComponent'] # useless
    df['Age_Water'] = df['AgeInDays'] / df['WaterComponent']
    df['Age_Cement'] = df['AgeInDays'] / df['CementComponent']
    df['Coarse_Fine'] = df['CoarseAggregateComponent'] / df['FineAggregateComponent']
    df['youngCementComponent'] = df.CementComponent * (df.AgeInDays < 40)
    df['youngSuperplasticizerComponent'] = df.SuperplasticizerComponent * (df.AgeInDays < 10)
    df['clippedAge'] = df.AgeInDays.clip(None, 40)
    df['clippedWater'] = df.WaterComponent.clip(195, None)
    df['hasBlastFurnaceSlag'] = df.BlastFurnaceSlag != 0
    df['hasFlyAshComponent'] = df.FlyAshComponent != 0
    df['hasSuperplasticizerComponent'] = df.SuperplasticizerComponent != 0
    

In [ ]:
class TargetEncoder(BaseEstimator, TransformerMixin):
    """Encodes the AgeInDays values by their average target value"""
    def fit(self, X, y):
        # VotingRegressor forwards y as an array
        if type(y) == np.ndarray:
            y = pd.Series(y, index=X.index)
        self.encodings_ = y.groupby(X['AgeInDays'].apply(self.replace_rare)).mean()
        return self
    
    def transform(self, X, y=None):
        X = X.copy()
        X['AgeInDays'] = self.encodings_.reindex(X['AgeInDays'].apply(self.replace_rare)).values
        return X

    @staticmethod
    def replace_rare(x):
        """Replace the rare AgeInDays values by nearby values"""
        if x == 1: return 3
        if x == 11: return 14
        if x == 49: return 56
#         if x == 91: return 90
#         if x == 120: return 100
        if x == 360: return 365
        return x



# Cross-validation

In [ ]:
def score_model(model, features_used, label=None):
    """Cross-validate a model with selected features"""
    n_repeats, n_splits = 5, 15
    features_used = [f for f in features_used if f != 'original']
    score_list = []
    oof = np.zeros_like(train[target])
    y_te_pred = np.zeros(len(test), dtype=float)
    kf = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=222)
    for fold, (idx_tr, idx_va) in enumerate(kf.split(train,
                                                     groups=train[original_features].apply(tuple, axis=1))):
        X_tr = train.iloc[idx_tr][features_used]
        X_va = train.iloc[idx_va][features_used]
        y_tr = train.iloc[idx_tr][target]
        y_va = train.iloc[idx_va][target]
        
#         X_tr = pd.concat([X_tr, original[features_used]], axis=0)
#         y_tr = pd.concat([y_tr, original[target]], axis=0)
        
        model.fit(X_tr, y_tr)
        trmse = mean_squared_error(y_tr, model.predict(X_tr), squared=False)
        y_va_pred = model.predict(X_va)
        rmse = mean_squared_error(y_va, y_va_pred, squared=False)
        if type(model) == Pipeline and type(model.steps[-1][1]) == Ridge:
            print('Weights:', model.steps[-1][1].coef_.round(2))
        print(f"Fold {fold}: trmse = {trmse:.3f}   rmse = {rmse:.3f}")
        oof[idx_va] += y_va_pred
        y_te_pred += model.predict(test[features_used])
        score_list.append(rmse)

    oof /= n_repeats
    y_te_pred /= n_repeats * n_splits
    rmse = sum(score_list) / len(score_list)
    print(f"Average rmse: {rmse:.3f} {label if label is not None else ''}")
    if label is not None:
        global result_list
        result_list.append((label, rmse, oof, y_te_pred))
    idxs = np.argsort(oof)
    oof_i = oof[idxs]
    y_true_i = train[target].iloc[idxs]
    s = pd.Series(y_true_i.values, index=oof_i)
    s = s.rolling(100, center=True).mean()
    plt.figure(figsize=(10, 4))
    plt.suptitle(label if label != None else '', fontsize=20)
    ax = plt.subplot(1, 2, 1) # y_true vs. y_pred
    plt.scatter(oof, train[target], s=3)
    plt.scatter(s.index, s, s=2, c='r')
    plt.plot([s.index.min(), s.index.max()], [s.index.min(), s.index.max()], c='y', lw='2')
    ax.set_aspect('equal')
    plt.xlabel('y_pred')
    plt.ylabel('y_true')
    plt.subplot(1, 2, 2) # histogram
    plt.hist(oof, bins=100, color='darkgreen')
    plt.xlabel('y_pred')
    plt.ylabel('count')
    plt.show()


In [ ]:
ridge_features = ['CementComponent', 'BlastFurnaceSlag', 'WaterComponent', 'SuperplasticizerComponent',
                     'CoarseAggregateComponent', 'FineAggregateComponent', 'AgeInDays', 'hasBlastFurnaceSlag',
                     'hasSuperplasticizerComponent', 'clippedWater', 'Coarse_Fine', 'clippedAge',
                     'youngCementComponent', 'youngSuperplasticizerComponent', 'original'
                    ]
score_model(model=make_pipeline(TargetEncoder(), StandardScaler(), Ridge(30)),
            features_used=ridge_features,
            label='Ridge')

In [ ]:
%%time
rf_features = original_features + ['Age_Water', 'Age_Cement']
score_model(model=make_pipeline(TargetEncoder(), RandomForestRegressor(n_estimators=300, min_samples_leaf=30, random_state=1)),
            features_used=rf_features,
            label='Random Forest')

In [ ]:
xgb_params = {'n_estimators': 700, 
              'max_depth': 8,
              'min_child_weight': 30,
              'learning_rate': 0.005,
              'base_score': train[target].mean(),
              'subsample': 0.25,
              'gamma': 10,
              'tree_method': 'hist',
              'random_state': 3,
              }
score_model(model=make_pipeline(TargetEncoder(), xgboost.XGBRegressor(**xgb_params)),
            features_used=original_features,
            label='XGBoost')


In [ ]:
cb_features = ['CementComponent', 'BlastFurnaceSlag',
               'SuperplasticizerComponent', 
               'CoarseAggregateComponent', 'AgeInDays', 'original']
cb_params = {
    'n_estimators': 550,
    'learning_rate': 0.01,
    'depth': 7,
    'random_strength': 0.5,
    'bagging_temperature': 0.7,
    'colsample_bylevel': 0.8,
    'l2_leaf_reg': 5,
    'verbose': False,
}
score_model(model=make_pipeline(TargetEncoder(),
                                catboost.CatBoostRegressor(**cb_params, random_state=1)),
            features_used=cb_features,
            label='CatBoost')


In [ ]:
gbr_params = {'n_estimators': 600,
              'max_depth': 4,
              'learning_rate': 0.01,
              'min_samples_leaf': 40 ,
              'max_features': 3}
score_model(model=make_pipeline(TargetEncoder(),
                                GradientBoostingRegressor(**gbr_params, random_state=2)),
            features_used=original_features,
            label='GradientBoostingRegressor')



In [ ]:
%%time
hgb_features = ['CementComponent', 'BlastFurnaceSlag',
                'SuperplasticizerComponent', 
                'CoarseAggregateComponent', 'AgeInDays', 'original']
score_model(model=make_pipeline(TargetEncoder(), 
                                HistGradientBoostingRegressor(learning_rate=0.01,
                                                              max_iter=1600,
                                                              max_leaf_nodes=3,
                                                              random_state=1)),
            features_used=hgb_features,
            label='HistGradientBoostingRegressor')


In [ ]:
%%time
lgbm_params = {
        'learning_rate': 0.0005,
        'n_estimators': 20000,
        'num_leaves': 7,
        'colsample_bytree': 0.4,
        'subsample': 0.5,
        'subsample_freq': 6,
        'min_child_samples': 25,
    }

score_model(model=lightgbm.LGBMRegressor(**lgbm_params, random_state=1),
            features_used=original_features,
            label='LGBM')


# Comparison

In [ ]:
result_list = [(a, b, c, d) for (a, b, c, d) in result_list if 'Mean' not in a]
result_df = pd.DataFrame(result_list, columns=['label', 'rmse', 'oof', 'y_te_pred'])
result_df.drop_duplicates(subset='label', keep='last', inplace=True)
result_df = result_df[~result_df.label.str.contains('Mean')]
result_df.sort_values('rmse', inplace=True)
result_df.reset_index(drop=True, inplace=True)

oof = np.column_stack(list(result_df.oof))
corr = np.corrcoef(oof, rowvar=False)
plt.figure(figsize=(5, 5))
plt.suptitle('Correlations', fontsize=20)
sns.heatmap(corr, linewidth=0.1, fmt='.3f', 
            annot=True, annot_kws={'size': 8}, 
            cmap='YlOrRd',
            xticklabels=result_df.label,
            yticklabels=result_df.label,
           )
plt.show()

r = result_df.copy()
for i in range(2, len(r)+1):
    oof = np.column_stack(list(r.oof.iloc[:i]))
    oof = oof.mean(axis=1)
    rmse = mean_squared_error(train[target], oof, squared=False)
    result_list.append((f"Mean of {i}", rmse, oof, None))

result_df = pd.DataFrame(result_list, columns=['label', 'rmse', 'oof', 'y_te_pred'])

result_df.drop_duplicates(subset='label', keep='last', inplace=True)
result_df.sort_values('rmse', inplace=True)
result_df.reset_index(drop=True, inplace=True)

plt.figure(figsize=(8, len(result_df) * 0.3))
plt.suptitle('Final comparison', fontsize=20)
bars = plt.barh(result_df.index, result_df.rmse, color='orange')
plt.gca().bar_label(bars, fmt='%.3f')
plt.gca().invert_yaxis()
plt.yticks(np.arange(len(result_df)), result_df.label)
plt.xticks(np.linspace(12.0, 12.2, 5))
plt.xlabel('RMSE')
plt.xlim(12, 12.2)
plt.show()


# Submission

In [ ]:
%%time
y_pred = np.column_stack(list(r.y_te_pred))
y_pred = y_pred.mean(axis=1)
pd.Series(y_pred, index=test.index, name=target).to_csv(f"submission.csv")
y_pred.round(1)